In [ ]:
import torch
print(torch.cuda.is_available())		 # 查看GPu设备是否可用
print(torch.cuda.device_count()) 		 # 查看GPu设备数量
print(torch.cuda.get_device_name())   	 # 查看当前GPu设备名称，默认设备id从0开始
print(torch.cuda.current_device())

/home/u4_3090_4/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


True
4
NVIDIA GeForce RTX 3090
0


In [2]:

import os
import random
import numpy as np
import torch
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
from utils.tools import dotdict
fix_seed = 2021
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

args = dotdict()
# 数据相关参数
args.root_path = '/home/u4_3090_4/dataset/ETT-small/'  # 数据集根目录
args.data_path = 'ETTh1.csv'            # 数据文件
args.model_id = 'ETTh1_672_96'          # 模型标识符
args.model = 'AutoTimes_Gpt2'          # 模型类型（基于Gpt2的AutoTimes）
args.data = 'ETTh1'                     # 数据集名称
args.features = 'M'                     # 特征类型（M=多变量预测多变量）

# 序列长度相关参数
args.seq_len = 672        # 输入序列长度（672个时间步）
args.label_len = 576      # 标签序列长度（576个时间步）
args.token_len = 96       # token长度（96个时间步）
args.test_pred_len = 96   # 测试时预测长度（96个时间步）
args.test_seq_len = 672   # 测试时输入序列长度（672个时间步）
args.test_label_len = 576 # 测试时标签序列长度（576个时间步）


# 训练相关参数
args.batch_size = 256      # 批次大小
args.learning_rate = 0.0005  # 学习率
args.train_epochs = 10     # 训练轮数
args.patience = 3          # 早停耐心值
args.loss = 'MSE'          # 损失函数（均方误差）
args.weight_decay = 0      # 权重衰减
args.lradj = 'type1'       # 学习率调整策略
args.use_amp = True           # 缺少了
args.cosine = True            # 缺少了
args.tmax = 10                # 缺少了
args.mix_embeds = False       # GPT2版本设为False
args.drop_last = False        # 设为False避免DataLoader问题
args.val_set_shuffle = True   # 缺少了
args.seasonal_patterns = 'Monthly'  # 缺少了

# 模型架构参数
args.mlp_hidden_layers = 0   # MLP隐藏层数
args.mlp_hidden_dim = 256    # MLP隐藏维度
args.mlp_activation = 'tanh' # 激活函数
args.dropout = 0.1           # dropout率

# 其他配置
args.llm_ckp_dir = '/home/u4_3090_4/baseModel'    # Gpt2模型检查点目录
args.checkpoints = '/home/u4_3090_4/checkpoints/'  # 模型保存目录
args.gpu = 0                   # GPU设备ID
# 在其他配置部分添加，cursor添加的
args.drop_short = False          # 是否丢弃过短序列，默认False
args.use_multi_gpu = False       # 是否使用多GPU，默认False  
args.local_rank = 0              # 本地GPU rank，默认0
args.visualize = True

# cursor增加
args.num_workers = 0
print('Args in experiment:')
print(args)

Args in experiment:
{'root_path': '/home/u4_3090_4/dataset/ETT-small/', 'data_path': 'ETTh1.csv', 'model_id': 'ETTh1_672_96', 'model': 'AutoTimes_Gpt2', 'data': 'ETTh1', 'features': 'M', 'seq_len': 672, 'label_len': 576, 'token_len': 96, 'test_pred_len': 96, 'test_seq_len': 672, 'test_label_len': 576, 'batch_size': 256, 'learning_rate': 0.0005, 'train_epochs': 10, 'patience': 3, 'loss': 'MSE', 'weight_decay': 0, 'lradj': 'type1', 'use_amp': True, 'cosine': True, 'tmax': 10, 'mix_embeds': False, 'drop_last': False, 'val_set_shuffle': True, 'seasonal_patterns': 'Monthly', 'mlp_hidden_layers': 0, 'mlp_hidden_dim': 256, 'mlp_activation': 'tanh', 'dropout': 0.1, 'llm_ckp_dir': '/home/u4_3090_4/baseModel', 'checkpoints': '/home/u4_3090_4/checkpoints/', 'gpu': 0, 'drop_short': False, 'use_multi_gpu': False, 'local_rank': 0, 'visualize': True, 'num_workers': 0}


In [ ]:

os.environ["CUDA_VISIBLE_DEVICES"] = '0'
exp = Exp_Long_Term_Forecast(args)
# setting record of experiments
setting = '{}_{}_{}_sl{}_ll{}_tl{}_lr{}_bt{}_wd{}_hd{}_hl{}_cos{}_mix{}_{}'.format(
    args.model_id,
    args.model,
    args.data,
    args.seq_len,
    args.label_len,
    args.token_len,
    args.learning_rate,
    args.batch_size,
    args.weight_decay,
    args.mlp_hidden_dim,
    args.mlp_hidden_layers,
    args.cosine,
    args.mix_embeds,
    args.des)
print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
exp.train(setting)
print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
exp.test(setting)
torch.cuda.empty_cache()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline
img = Image.open('test_results/ETTh1_672_96_AutoTimes_Llama_ETTh1_sl672_ll576_tl96_lr0.0005_bt256_wd0_hd256_hl0_cosTrue_mixTrue_test/96/0.png')
# 使用matplotlib显示图片
plt.imshow(img)
plt.axis('off') 
plt.show()